In [1]:
import os
#import numpy as np
import pandas as pd
import torch
from torch import nn, optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from torchsummary import summary
import matplotlib.pyplot as plt
from PIL import Image

In [2]:
data_dir = "/kaggle/input/flowers/"
train_dir = os.path.join(data_dir, "Training Data/Training Data") 
valid_dir = os.path.join(data_dir, "Validation Data/Validation Data")
test_dir = os.path.join(data_dir, "Testing Data/Testing Data")

In [3]:
#transformation (ONLY use data augmentation only on training to avoid overfitting)
Image_size = (224, 224) #most pretrained CNNs were trained on ImageNet where all images were resized to 224x224

train_transforms = transforms.Compose([
    transforms.Resize(Image_size),                          #to resize all images to be the same
    transforms.RandomHorizontalFlip(),                      #for diversity
    transforms.RandomRotation(20),                          #for diversity
    transforms.ToTensor(),                                  #convert to Tensor
    transforms.Normalize([0.485, 0.456, 0.406], 
                         [0.229, 0.224, 0.225])])           #match ImageNet stats

valid_transforms = transforms.Compose([
    transforms.Resize(Image_size),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], 
                         [0.229, 0.224, 0.225])])

In [4]:
train_dataset = datasets.ImageFolder(train_dir, transform=train_transforms)
valid_dataset = datasets.ImageFolder(valid_dir, transform=valid_transforms)
test_dataset  = datasets.ImageFolder(test_dir,  transform=valid_transforms)

BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") 
device

device(type='cuda')

In [6]:
#compared various pretrained models (DenseNet, MobileNetV2, EfficientNet-B0, ResNet-10) and chose 
#EfficientNet-B0: 237-layers, laptop friendly, middle ground of DenseNet and MobileNetv2. Better accuracy then MobileNetV2
#avoid overfitting, works great with small dataset, and fast training.

model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False       #freeze all parameters

num_classes= 10
model.classifier[1] = nn.Linear(in_features= 1280, out_features = num_classes) #replace the last layer
model = model.to(device)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 176MB/s]


In [7]:
from torchsummary import summary 

summary(model, (3, 224, 224))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 32, 112, 112]             864
       BatchNorm2d-2         [-1, 32, 112, 112]              64
              SiLU-3         [-1, 32, 112, 112]               0
            Conv2d-4         [-1, 32, 112, 112]             288
       BatchNorm2d-5         [-1, 32, 112, 112]              64
              SiLU-6         [-1, 32, 112, 112]               0
 AdaptiveAvgPool2d-7             [-1, 32, 1, 1]               0
            Conv2d-8              [-1, 8, 1, 1]             264
              SiLU-9              [-1, 8, 1, 1]               0
           Conv2d-10             [-1, 32, 1, 1]             288
          Sigmoid-11             [-1, 32, 1, 1]               0
SqueezeExcitation-12         [-1, 32, 112, 112]               0
           Conv2d-13         [-1, 16, 112, 112]             512
      BatchNorm2d-14         [-1, 16, 1

In [8]:
def train_model(model, criterion, optimizer, train_loader, valid_loader, device, epochs=10, verbose=True):
    train_loss, valid_loss, valid_accuracy = [], [], []

    for epoch in range(epochs):
        model.train()
        train_batch_loss = 0
        valid_batch_loss = 0
        valid_batch_acc = 0
        
        # Training loop
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            y_hat = model(X)
            loss = criterion(y_hat, y)
            loss.backward()
            optimizer.step()
            train_batch_loss += loss.item()
        train_loss.append(train_batch_loss / len(train_loader))
        
        # Validation Loop
        model.eval()
        with torch.no_grad():
            for X, y in valid_loader:
                X, y = X.to(device), y.to(device)
                y_hat = model(X)
                _, y_hat_labels = torch.softmax(y_hat, dim=1).topk(1, dim=1)
                loss = criterion(y_hat, y)
                valid_batch_loss += loss.item()
                valid_batch_acc += (y_hat_labels.squeeze() == y).type(torch.float32).mean().item()
                
        valid_loss.append(valid_batch_loss / len(valid_loader))
        valid_accuracy.append(valid_batch_acc / len(valid_loader))
                
        if verbose:
            print(f"Epoch {epoch + 1}:",
                  f"Train Loss: {train_loss[-1]:.3f}.",
                  f"Valid Loss: {valid_loss[-1]:.3f}.",
                  f"Valid Accuracy: {valid_accuracy[-1]:.2f}.")
    results = {"train_loss": train_loss,
               "valid_loss": valid_loss,
               "valid_accuracy": valid_accuracy}
    return results

In [9]:
criterion = nn.CrossEntropyLoss()     #multiclass Loss function
optimizer = torch.optim.Adam(
    model.classifier.parameters(),
    lr = 0.0001)

In [10]:
train_model(model, criterion, optimizer, train_loader, valid_loader, device, epochs=10)

Epoch 1: Train Loss: 1.860. Valid Loss: 1.532. Valid Accuracy: 0.68.
Epoch 2: Train Loss: 1.355. Valid Loss: 1.188. Valid Accuracy: 0.72.
Epoch 3: Train Loss: 1.143. Valid Loss: 1.050. Valid Accuracy: 0.74.
Epoch 4: Train Loss: 1.030. Valid Loss: 0.950. Valid Accuracy: 0.75.
Epoch 5: Train Loss: 0.968. Valid Loss: 0.901. Valid Accuracy: 0.76.
Epoch 6: Train Loss: 0.918. Valid Loss: 0.853. Valid Accuracy: 0.76.
Epoch 7: Train Loss: 0.887. Valid Loss: 0.823. Valid Accuracy: 0.77.
Epoch 8: Train Loss: 0.858. Valid Loss: 0.783. Valid Accuracy: 0.77.
Epoch 9: Train Loss: 0.837. Valid Loss: 0.763. Valid Accuracy: 0.78.
Epoch 10: Train Loss: 0.827. Valid Loss: 0.768. Valid Accuracy: 0.78.


{'train_loss': [1.8597722223826818,
  1.3550910967499463,
  1.1428172393902531,
  1.030154402957542,
  0.9679934705244199,
  0.9175908205859951,
  0.8867414385270971,
  0.8577159586618704,
  0.837478659681674,
  0.8270192382686428],
 'valid_loss': [1.5320801594454772,
  1.187588247903593,
  1.0497290691372696,
  0.9502807126683035,
  0.9008396603878895,
  0.8533581826527408,
  0.8225172751458587,
  0.783123899037671,
  0.7634245272085165,
  0.7684194262430166],
 'valid_accuracy': [0.6793391719745223,
  0.7229299363057324,
  0.7396496815286624,
  0.7531847133757962,
  0.7565684713375797,
  0.7641321656050956,
  0.7708996815286624,
  0.7726910828025477,
  0.7792595541401274,
  0.7760748407643312]}

In [11]:
model.eval()   # set model to evaluation mode

correct = 0
total = 0

with torch.no_grad():
    for X, y in test_loader:
        X, y = X.to(device), y.to(device)

        outputs = model(X)
        _, preds = torch.max(outputs, 1)

        correct += (preds == y).sum().item()
        total += y.size(0)

test_accuracy = correct / total
print("Test accuracy:", test_accuracy)

Test accuracy: 0.7927813163481954


In [12]:
class_names= train_dataset.classes
class_names

['Aster',
 'Daisy',
 'Iris',
 'Lavender',
 'Lily',
 'Marigold',
 'Orchid',
 'Poppy',
 'Rose',
 'Sunflower']